# Epsilon Fund — XS Momentum (`xs_1_2_i_r2_d`) CPCV Validation

Combinatorial Purged Cross-Validation for `xs_1_2_i_r2_d`, run via
`infrastructure/walkforward/cpcv_engine.py`.

This is the **rolling-Sharpe** sibling of `xs_2.ipynb` (which uses residual
Sharpe).  Same Lee-Swaminathan two-stage + inverse-vol + Layer 2 breadth +
Layer 3 dispersion + asymmetric K_long/K_short — only the ranking signal
differs:

| Family | Signal | Warmup |
|---|---|---|
| `xs_3` (residual Sharpe) | strip BTC β, then J-bar Sharpe of residuals | `2J + 1` |
| **`xs_1`** (rolling Sharpe) | J-bar Sharpe of raw returns (no β stripping) | `J + 1` |

Empirically rolling Sharpe is faster, cheaper, and often within ~0.1 Sharpe of
residual Sharpe in crypto where BTC β shifts rapidly.

---
### Strategy summary (matches `xs_1_2_i_r2_d.ipynb`)

- **Signal:** **rolling Sharpe** of raw returns over J bars (vol-normalised momentum, no β stripping).
- **Universe:** dynamic perp filter (top_n=30, min_volume=$80M, min_age=90d).
- **Selection:** Lee-Swaminathan two-stage —
  * Stage 1: rank by composite, keep top/bottom `pct × pool_multiplier`.
  * Stage 2: refine to `pct` by HIGHEST `vol_short_window` / `vol_long_window` volume-change.
  * Both legs use the pool-refine path (`volume_filter_legs='both'`).
- **Intra-leg weighting:** inverse-volatility (`iv_vol_window` Optuna-swept).
- **Layer 1 regime (skipped):** ADX-scaled tilt — replaced by Layer 2.
- **Layer 2 regime:** BTC + universe **breadth** tilt with **SMA** discriminator
  (`breadth_ma_period` Optuna-swept).
- **Layer 3 dispersion gate:** all parameters Optuna-swept per trial.
- **Asymmetric holding:** `K_long` and `K_short` swept independently.

---
### Workflow

1. Build / refresh the perp universe cache: `python topics/momentum/xs_momentum/universe/universe_cache.py`.
2. Set `WF_SHARPE` to the combined OOS Sharpe from `xs_1_2_i_r2_d.ipynb` for the comparison annotation.
3. Run all cells.


In [ ]:
import sys
import os
import pandas as pd
import numpy as np


# ── repo root — works on both Mac and Windows ────────────────────────────────
ROOT = os.path.expanduser('~/Desktop/epsilon/github/Epsilon-Quant-Research')
# ROOT = r'C:\\Users\\user\\Documents\\Epsilon Fund\\Epsilon-Quant-Research'  # ← Windows path
# ─────────────────────────────────────────────────────────────────────────────

sys.path.insert(0, os.path.join(ROOT, 'infrastructure', 'data'))
sys.path.insert(0, os.path.join(ROOT, 'infrastructure', 'walkforward'))
sys.path.insert(0, os.path.join(ROOT, 'infrastructure', 'backtester'))
sys.path.insert(0, os.path.join(ROOT, 'topics', 'momentum', 'xs_momentum'))
sys.path.insert(0, os.path.join(ROOT, 'topics', 'momentum', 'xs_momentum', 'universe'))


from binance_client import get_binance_client, get_data
from cpcv_engine import (
    run_cpcv,
    cpcv_parameter_analysis,
    cpcv_print_param_suggestions,
    cpcv_summary,
    cpcv_confidence_intervals,
    cpcv_ci_summary,
)
from cpcv_visualizer import (
    plot_cpcv_results,
    plot_parameter_distributions,
    plot_parameter_correlation_matrix,
    plot_split_performance_heatmap,
    plot_tercile_comparison,
)
from xs_strategy import (
    make_xs_strategy,
    compute_btc_regime_breadth_tilt_panel,
    compute_dispersion_confidence_panel,
    summarize_tilt_distribution,
)
from ls_diagnostics  import (compute_attribution, plot_attribution,
                             bear_hedge_diagnostic, regime_quadrant_diagnostic)
from universe_filter import load_cache

---
## Data + Universe Selection (FIXED constants)

The four `UF_*` parameters below are **structural design choices**, not
hyperparameters.  They control which coins are eligible to enter the long /
short legs at each rebalance.  Optuna does NOT search over these — keeping the
universe stable across CPCV splits is what makes the parameter-distribution
and tercile diagnostics interpretable.

| Constant | What it controls |
|---|---|
| `UF_TOP_N` | Maximum coins in the eligible universe at each rebalance |
| `UF_MIN_VOLUME` | Minimum 30-day avg daily USDT perp volume in dollars |
| `UF_MIN_AGE_DAYS` | Minimum days since the perp listing date |
| `UF_VOLUME_WINDOW` | Rolling window for the volume average |

Strategy parameters (`J`, `K_long`, `K_short`, `pct_long`, `pct_short`,
`breadth_ma_period`) go in `PARAM_DEFS` further down and are
Optuna-optimisable.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# UNIVERSE FILTER CONFIG  (FIXED — not Optuna-optimisable)
# ══════════════════════════════════════════════════════════════════════════════
UF_TOP_N          = 30          # max coins in the eligible universe at each rebalance
UF_MIN_VOLUME     = 80_000_000  # min 30-day avg daily USDT perp volume ($)
UF_MIN_AGE_DAYS   = 90          # min days since perp listing date
UF_VOLUME_WINDOW  = 30          # rolling window for avg volume computation

# ══════════════════════════════════════════════════════════════════════════════
# DATA WINDOW
# ══════════════════════════════════════════════════════════════════════════════
LOOKBACK = 2200    # days; covers ~6 years of daily bars
COST     = 0.001   # per-leg trading cost fraction (baked into strategy_returns)

# ══════════════════════════════════════════════════════════════════════════════
# Inverse-vol weighting (intra-leg risk parity) — STRUCTURAL constant
# ══════════════════════════════════════════════════════════════════════════════
IV_VOL_WINDOW = 30

# ══════════════════════════════════════════════════════════════════════════════
# Asymmetric tilt magnitude caps — STRUCTURAL constants
# ══════════════════════════════════════════════════════════════════════════════
# Tilt magnitudes are CAPPED here.  Optuna optimises the tilt SCHEDULE
# (breadth_ma_period) below, not the magnitudes.
MAX_BULL_TILT = 0.20
MAX_BEAR_TILT = 0.20

# ══════════════════════════════════════════════════════════════════════════════
# Layer 2 — BTC + universe BREADTH tilt panel
# ══════════════════════════════════════════════════════════════════════════════
# Direction: BTC > MA(period) → bull, else bear (binary).
# Intensity: fraction of universe coins on the same side of their MA(period)
#            as BTC is on its (∈ [0, 1]).
# Tilt = direction × breadth × asymmetric MAX.
#
# This notebook uses **SMA** for the regime discriminator (academic regime-
# classification convention; smoother than EMA, fewer whipsaws).  Optuna
# sweeps `breadth_ma_period` over a wide [30, 200] band — SMA's natural
# calibration is longer than EMA's.
BREADTH_MA_TYPE    = 'sma'
BREADTH_MA_DEFAULT = 100   # used only for the diagnostic distribution printout

# ══════════════════════════════════════════════════════════════════════════════
# Layer 3 — Dispersion gate (signal-quality-driven gross-exposure scaling)
# ══════════════════════════════════════════════════════════════════════════════
# Per-bar conviction multiplier on `strategy_returns`, derived from
# cross-sectional return dispersion.  High dispersion → universe is
# differentiated → ranking signal informative → trade closer to baseline.
# Low dispersion → everything moves together → ranking is noise → scale down.
#
# Rolling 252-bar quantiles are recomputed point-in-time (no lookahead).
# Linear ramp between (q_low, q_high) maps to (CONFIDENCE_LOW, CONFIDENCE_HIGH).
DISPERSION_ROLLING_WINDOW = 252
DISPERSION_MIN_PERIODS    = 60
DISPERSION_QUANTILES      = (0.20, 0.80)
CONFIDENCE_LOW            = 0.5
CONFIDENCE_HIGH           = 1.5
# ══════════════════════════════════════════════════════════════════════════════
# Lee-Swaminathan two-stage selection — STRUCTURAL constant
# ══════════════════════════════════════════════════════════════════════════════
# pool_multiplier × pct = candidate pool size before vol-change refinement.
# 2.0 is the academic default (Lee-Swaminathan 2000); raises to 2.5–3.0 add
# selection breadth at the cost of weaker composite-signal coverage.
POOL_MULTIPLIER = 2.0


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Load multi-asset perp panel from cache + BTC OHLC for diagnostics
# ══════════════════════════════════════════════════════════════════════════════
print('Loading perp cache...')
close_full, volume, meta = load_cache()
print(f'  close  : {close_full.shape}')
print(f'  volume : {volume.shape}')
print(f'  meta   : {meta.shape}')

# Clip price panel to LOOKBACK days; volume + meta keep full history so the
# universe filter's rolling 30-day avg has enough bars even when as_of_date
# sits at the start of LOOKBACK.
cutoff     = close_full.index[-1] - pd.Timedelta(days=LOOKBACK)
close_full = close_full.loc[close_full.index >= cutoff]

# Restrict panel to coins with a recorded listing date
tradeable = sorted([c for c in close_full.columns
                    if c in meta.index and pd.notna(meta.loc[c, 'listing_date'])])
panel = close_full[tradeable]

# Equal-weight basket benchmark + driver_df (BTC supplies the calendar)
ew_equity = (1 + panel.pct_change().mean(axis=1).fillna(0)).cumprod()
ew_df     = pd.DataFrame({'Close': ew_equity}, index=panel.index)
driver_df = panel[['BTC']].rename(columns={'BTC': 'Close'}).copy()

print(f'\nTradeable universe: {len(tradeable)} coins')
print(f'Date range: {panel.index[0].date()} → {panel.index[-1].date()}  '
      f'({len(panel)} bars, clipped to LOOKBACK={LOOKBACK})')

# BTC close for the breadth regime classifier (perp cache stores Close already,
# but pull OHLC fresh so the bear-hedge / regime-quadrant diagnostics later
# have High/Low — they don't strictly need them but cpcv_template uses them).
client = get_binance_client()
print('\nFetching BTC OHLC for diagnostics...')
btc_ohlc  = get_data(client, 'BTCUSDT', '1d', LOOKBACK)
btc_close = btc_ohlc['Close']
print(f'  BTC OHLC: {btc_ohlc.shape}  '
      f'({btc_ohlc.index[0].date()} → {btc_ohlc.index[-1].date()})')

---
## Layer 2 — Breadth Tilt Panel (Diagnostic)

Build a default breadth tilt panel using `BREADTH_MA_DEFAULT` to sanity-check
the regime distribution before launching CPCV.  The factory will recompute
this panel **per Optuna trial** using each trial's `breadth_ma_period`, with
caching by `(mode, ma_type, period)` so identical trials reuse the same panel.


In [ ]:
regime_tilt_panel = compute_btc_regime_breadth_tilt_panel(
    btc_close,
    universe_panel = panel,
    ma_period      = BREADTH_MA_DEFAULT,
    ma_type        = BREADTH_MA_TYPE,
    max_bull_tilt  = MAX_BULL_TILT,
    max_bear_tilt  = MAX_BEAR_TILT,
)

summarize_tilt_distribution(regime_tilt_panel,
                             max_bull_tilt = MAX_BULL_TILT,
                             max_bear_tilt = MAX_BEAR_TILT,
                             btc_adx       = None)

print(f'\nThe factory will recompute the breadth tilt panel per-Optuna-trial '
      f'using params[\'breadth_ma_period\'] (range from PARAM_DEFS).')
print(f'BREADTH_MA_DEFAULT = {BREADTH_MA_DEFAULT} (used only for the above '
      f'distribution summary).')

---
## Layer 3 — Dispersion Confidence Panel

Computed once on the full panel and passed into the factory as
`confidence_panel`.  Inside the strategy_fn this is reindexed onto each
CPCV slice's date range and applied bar-wise — strictly causal because the
quantiles are rolling, not fixed.


In [ ]:
confidence_panel, dispersion_panel = compute_dispersion_confidence_panel(
    panel,
    rolling_window  = DISPERSION_ROLLING_WINDOW,
    min_periods     = DISPERSION_MIN_PERIODS,
    q_low           = DISPERSION_QUANTILES[0],
    q_high          = DISPERSION_QUANTILES[1],
    confidence_low  = CONFIDENCE_LOW,
    confidence_high = CONFIDENCE_HIGH,
)

print(f'Dispersion panel: {dispersion_panel.shape[0]} bars')
print(f'  Dispersion stats:'
      f'  min={dispersion_panel.min():.4f}'
      f'  median={dispersion_panel.median():.4f}'
      f'  max={dispersion_panel.max():.4f}')

print(f'\nConfidence multiplier distribution (after rolling-quantile mapping):')
for label, q in [('p10', 0.10), ('p25', 0.25), ('median', 0.50),
                 ('p75', 0.75), ('p90', 0.90)]:
    print(f'  {label:>6}: {confidence_panel.quantile(q):.3f}')
pct_at_high = (confidence_panel >= CONFIDENCE_HIGH - 1e-9).mean() * 100
pct_at_low  = (confidence_panel <= CONFIDENCE_LOW  + 1e-9).mean() * 100
scaled_down = (confidence_panel < 1.0 - 1e-9).mean() * 100
print(f'\n  Bars at floor ({CONFIDENCE_LOW}): {pct_at_low:.1f}%')
print(f'  Bars at cap   ({CONFIDENCE_HIGH}): {pct_at_high:.1f}%')
print(f'  Bars with confidence < 1.0 (scaled-down exposure): {scaled_down:.1f}%')

---
## Strategy Parameters

Define what Optuna searches (`PARAM_DEFS`) and what's locked (`FIXED_PARAMS`).
Universe selection and structural Layer-1/2/3 constants are fixed above.

| Param | Type | Range | Notes |
|---|---|---|---|
| `J` | int | 7–45 | Formation / lookback for residual-Sharpe signal |
| `K_long` | int | 7–21 | Long-leg holding period |
| `K_short` | int | 7–21 | Short-leg holding period (expect shorter optimum: shorts decay faster) |
| `pct_long` | float | 0.05–0.30 | Fraction of eligible universe in the long leg |
| `pct_short` | float | 0.05–0.30 | Fraction of eligible universe in the short leg |
| `breadth_ma_period` | int | 30–200 | SMA span for the breadth discriminator (BTC + each coin) |

Move any param that converges across splits (CV < 0.15 in `cpcv_print_param_suggestions`)
into `FIXED_PARAMS` to speed up subsequent runs.


In [ ]:
# ── search ranges ─────────────────────────────────────────────────────────────
# Format: 'param_name': ('int' | 'float', lo, hi)
#
# xs_3_2_i_r2_d-specific params (Lee-Swaminathan Stage 2):
#   vol_short_window — recent attention window
#   vol_long_window  — baseline attention window
#   Strategy enforces vol_short_window < vol_long_window; on violation the
#   refinement is skipped (defensive).
#
# Layer-3 dispersion + iv_vol_window are per-trial (factory caches them).
PARAM_DEFS = {
    'J':                  ('int',   7,    45),
    'K_long':             ('int',   7,    21),
    'K_short':            ('int',   7,    21),
    'pct_long':           ('float', 0.05, 0.25),
    'pct_short':          ('float', 0.05, 0.25),
    'breadth_ma_period':  ('int',   30,   200),

    # Lee-Swaminathan Stage 2 volume-change windows (xs_3_2 family)
    'vol_short_window':   ('int',   3,    10),
    'vol_long_window':    ('int',   20,   60),

    # Intra-leg inverse-vol weighting lookback (days)
    'iv_vol_window':      ('int',   10,   60),

    # Layer 3 — dispersion gate
    'dispersion_window':  ('int',   60,   504),
    'confidence_low':     ('float', 0.0,  0.8),
    'confidence_high':    ('float', 1.0,  2.0),
}

# ── anchored params ───────────────────────────────────────────────────────────
# Paste any param here that converged across CPCV splits (CV<0.15).
FIXED_PARAMS = {
    # 'pct_long': 0.20,
}


---
## Strategy Factory

`make_xs_strategy(...)` is a **closure factory** that captures the perp panel,
universe filter, regime mode, and dispersion confidence panel.  The returned
`strategy_fn(df_slice, params)` is what the CPCV engine calls — `df_slice` is
used only for its `DatetimeIndex` (BTC's calendar), and `params` carries the
Optuna trial's J / K_long / K_short / pct_long / pct_short / breadth_ma_period.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Strategy factory (infrastructure/walkforward/xs_strategy.py)
# ══════════════════════════════════════════════════════════════════════════════
# `signal_kind='rolling_sharpe'` swaps the residual-Sharpe ranker for plain
# rolling Sharpe of raw returns — xs_1 family.
# `confidence_panel` not passed — factory builds per-trial from params.
# `iv_vol_window` factory arg is the default; per-trial value comes from params.
my_strategy = make_xs_strategy(
    panel             = panel,
    volume            = volume,
    meta              = meta,
    uf_top_n          = UF_TOP_N,
    uf_min_volume     = UF_MIN_VOLUME,
    uf_min_age        = UF_MIN_AGE_DAYS,
    uf_volume_window  = UF_VOLUME_WINDOW,
    iv_vol_window     = IV_VOL_WINDOW,

    signal_kind       = 'rolling_sharpe',  # xs_1 family — Sharpe of raw returns

    # Lee-Swaminathan two-stage: composite pool → vol-change refinement
    pool_multiplier   = POOL_MULTIPLIER,
    volume_filter_legs = 'both',           # both legs pool-refined (xs_1_2 convention)

    # Layer 2 — breadth-based regime tilt (replaces ADX from Layer 1)
    btc_close              = btc_close,
    universe_breadth_panel = panel,
    regime_mode            = 'breadth',
    breadth_ma_type        = BREADTH_MA_TYPE,
    max_bull_tilt          = MAX_BULL_TILT,
    max_bear_tilt          = MAX_BEAR_TILT,

    # Layer 3 — dispersion gate now per-trial; no pre-computed panel.
)


---
## CPCV Config + Scoring

Partitions the data into `N` equal groups and optimises on every `C(N, k)`
training complement.

| Parameter | Default |
|---|---|
| `N` (groups) | `8` |
| `K_HOLD` (groups held out) | `2` |
| `N_TRIALS` (Optuna trials per split) | `300` |
| `BURNIN` (bars prepended to each test group) | `50` |
| `PURGE_BARS` | `1` |

**Burnin = 50** is enough for `J + 1` rolling-Sharpe warmup at max `J = 45` (46);
50 leaves a small safety margin.  This is HALF the 95-bar burnin used by the
`xs_3` (residual-Sharpe) family — a small efficiency win since rolling Sharpe
only needs one rolling window, not two compounded ones.

> **Runtime:** `C(N, k) × N_TRIALS` Optuna evaluations.
> For N=8, k=2, N_TRIALS=300: ~8,400 backtests.


In [ ]:
N          = 8     # groups to partition data into  →  C(8,2) = 28 splits
K_HOLD     = 2     # groups held out per split
N_TRIALS   = 300   # Optuna trials per split (match WF notebook)
BURNIN     = 50    # J + 1 for the rolling-Sharpe signal at max J=45
PURGE_BARS = 1

# ── walk-forward Sharpe for comparison annotation (optional) ──────────────────
WF_SHARPE  = None   # ← set to combined OOS Sharpe from xs_1_2_i_r2_d.ipynb


# ── SCORING FUNCTION ─────────────────────────────────────────────────────────
# Match the WF notebook's score_fn so CPCV ranks splits the same way the WF
# folds were ranked — keeps parameter-distribution / tercile output comparable.
def SCORE_FN(m):
    SHARPE_MAX = 3.0
    CALMAR_MAX = 6.0
    RETURN_MAX = 10.0
    calmar = (m['total_return'] / abs(m['max_drawdown'])
              if m['max_drawdown'] != 0 else 0.0)
    s = np.clip(m['sharpe_ratio']  / SHARPE_MAX, 0, 1)
    c = np.clip(calmar             / CALMAR_MAX, 0, 1)
    r = np.clip(m['total_return']  / RETURN_MAX, 0, 1)
    return 0.60 * s + 0.25 * c + 0.15 * r


# ── REJECTION CRITERIA ───────────────────────────────────────────────────────
# Default reject_fn checks num_trades < 7 → would reject every trial because
# this strategy uses position=1 everywhere (precomputed-returns path).  Filter
# on Sharpe / drawdown / total return instead.
def REJECT_FN(m):
    if m is None:                   return True
    if m['sharpe_ratio'] < 0.0:     return True
    if m['max_drawdown'] < -0.80:   return True
    if m['total_return'] < 0.0:     return True
    return False

In [ ]:
results = run_cpcv(
    df           = driver_df,         # BTC supplies the calendar; strategy_fn ignores its columns
    strategy_fn  = my_strategy,
    param_defs   = PARAM_DEFS,
    fixed_params = FIXED_PARAMS,
    N            = N,
    k            = K_HOLD,
    purge_bars   = PURGE_BARS,
    n_trials     = N_TRIALS,
    burnin       = BURNIN,
    cost         = 0.0,               # costs already baked into strategy_returns
    score_fn     = SCORE_FN,
    reject_fn    = REJECT_FN,
    verbose      = True,
)

### CPCV Summary — display flags

`cpcv_summary` is split into two cells so the output stays readable.  Each
flag controls one section of the printed report.


In [ ]:
# Group legend + path distribution percentile table + CI
cpcv_summary(
    results,
    show_group_legend = False,
    show_distribution = True,
    show_paths        = False,
    show_highlights   = False,
    show_split_legend = False,
    show_ci           = True,
)

In [ ]:
# Top/bottom highlights + split legend
cpcv_summary(
    results,
    show_group_legend = False,
    show_distribution = False,
    show_paths        = False,
    show_highlights   = True,
    show_split_legend = True,
    show_ci           = False,
)

---
## Path-Level Charts

- **Equity curves** — all 105 paths (semi-transparent), the mean path (bold),
  and the min–max envelope.  Vertical dotted lines mark group boundaries.
- **Sharpe histogram** — distribution of the 105 path-level Sharpe ratios with
  the overlap-adjusted CI overlay.


In [ ]:
ci = cpcv_confidence_intervals(results)
plot_cpcv_results(results, wf_sharpe=WF_SHARPE, ci_results=ci);

---
## Parameter Analysis

Compute the analysis object once — all charts below read from it.


In [ ]:
analysis = cpcv_parameter_analysis(results)

# Prints consensus_ranges table + copy-pasteable PARAM_DEFS / FIXED_PARAMS blocks.
# CV < 0.10  → "fix at {median}"   : converges consistently, safe to anchor.
# CV 0.10–0.25 → "narrow to IQR"   : reduce search range to Q25–Q75 next CPCV run.
# CV > 0.25  → "keep current range": period-sensitive, keep free.
cpcv_print_param_suggestions(results, analysis)

### Parameter Distributions

One subplot per free parameter.  Each dot is one split's best value, jittered
horizontally for readability and **coloured by that split's OOS Sharpe**
(red = low, green = high).  For this strategy, watch `K_long` vs `K_short`
clustering — if they separate (K_short < K_long), the asymmetry thesis is
working.  Watch `breadth_ma_period` for stability across splits — it typically
clusters near the long end (150–200) when SMA is used.


In [ ]:
plot_parameter_distributions(results, analysis=analysis);

### Parameter Correlation Matrix

Pearson correlation between every pair of free parameters across the 28
splits.  |r| > 0.6 means the optimizer is sliding along a ridge — consider
fixing the ratio or removing one parameter.


In [ ]:
plot_parameter_correlation_matrix(analysis);

### Split Performance Heatmap

N × N grid where cell (i, j) shows the OOS Sharpe of the split that held out
groups i and j.  Persistent red cells around a particular group index means
that calendar period is structurally hard for the L/S signal — cross-reference
the group date legend to identify the regime.


In [ ]:
plot_split_performance_heatmap(results);

### Tercile Comparison

Each subplot: parameter values chosen by the top vs bottom Sharpe tercile of
splits.  Separation > 1 means parameter choice is predictive of OOS
performance — anchor toward the top-tercile median or narrow the search range.


In [ ]:
plot_tercile_comparison(results, analysis);

---
## XS-Specific Diagnostics — Consensus-Param Backtest

Re-run the strategy on the FULL date range using the consensus parameters
(median across CPCV splits) and compute the long/short attribution + bear-hedge
+ regime-quadrant diagnostics.  Same OOS diagnostics the WF notebook prints,
but applied to the cross-validated consensus rather than to a single WF schedule.


In [ ]:
# Build consensus params = median across splits, cast to the right dtype.
# (results['param_distributions'] is a DataFrame: rows=splits, cols=free params)
param_dist = results['param_distributions']
consensus  = {**FIXED_PARAMS}
for name, (dtype, _, _) in PARAM_DEFS.items():
    if name in FIXED_PARAMS or name not in param_dist.columns:
        continue
    median = param_dist[name].median()
    consensus[name] = int(round(median)) if dtype == 'int' else float(median)

print('Consensus params (median across CPCV splits):')
for k, v in consensus.items():
    print(f'  {k:<22}  {v}')

# Build a full-range slice with the same OHLC shape strategy_fn expects
full_slice = driver_df.copy()
oos_full, _ = my_strategy(full_slice, consensus)

# Trim warmup so the diagnostic only sees bars where the strategy was active
oos = oos_full.iloc[BURNIN:].copy()
print(f'\nConsensus-path OOS span: {oos.index[0].date()} -> {oos.index[-1].date()}  '
      f'({len(oos)} bars)')


In [ ]:
attr = compute_attribution(oos, ann=365)
print('═' * 60)
print('LONG/SHORT ATTRIBUTION (consensus path)')
print('═' * 60)
for k, v in attr.items():
    if isinstance(v, float):
        if 'sharpe' in k:
            print(f'  {k:<22} {v:>8.3f}')
        elif 'return' in k or 'total' in k or 'rate' in k or 'turnover' in k:
            print(f'  {k:<22} {v:>8.2%}' if abs(v) < 100 else f'  {k:<22} {v:>8.3f}')
        else:
            print(f'  {k:<22} {v:>8.2f}')
    else:
        print(f'  {k:<22} {v:>8}')

plot_attribution(oos, benchmark_data=ew_df, show=True, save_html=None,
                 title='XS Momentum xs_1_2_i_r2_d — L/S Diagnostics (CPCV consensus path)')

In [ ]:
bh = bear_hedge_diagnostic(oos, panel['BTC'], ann=365)

In [ ]:
rq = regime_quadrant_diagnostic(oos, panel['BTC'],
                                ma_window=200, er_window=14, ann=365)

---
## Save Results

Pickle the full results dict for portfolio-level analysis and future reference.


In [ ]:
os.makedirs('oos', exist_ok=True)
results_file = os.path.join('oos', 'xs_1_2_i_r2_d_cpcv.pkl')
pd.to_pickle(results, results_file)
print(f'Saved CPCV results to {results_file}')

# Optional: save the consensus-path OOS dataframe for portfolio integration
oos_file = os.path.join('oos', 'xs_1_2_i_r2_d_cpcv_consensus_oos.pkl')
oos.to_pickle(oos_file)
print(f'Saved consensus-path OOS dataframe to {oos_file}')